[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/20_weight_init.ipynb)

# 🟢 简单：Kaiming 初始化

实现权重张量的 **Kaiming（He）正态初始化**。

$$W \sim \mathcal{N}(0, \text{std}^2) \quad \text{其中} \quad \text{std} = \sqrt{\frac{2}{\text{fan\_in}}}$$

### 函数签名
```python
def kaiming_init(weight: Tensor) -> Tensor:
    # 原地用 Kaiming 正态分布初始化 weight
    # fan_in = weight.shape[1]
    # 返回 weight 张量
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✏️ 在此实现你的代码

def kaiming_init(weight):
    pass  # 用 normal(0, sqrt(2/fan_in)) 填充

fan_in 和 fan_out 的定义

- 对于全连接层 (Linear Layer)
- weight形状: [out_features, in_features]
- 例如: \
Linear(256, 128)  # 输入256，输出128\
weight = torch.empty(128, 256)\
fan_in = 256   # 输入特征数\
fan_out = 128  # 输出特征数

- 对于卷积层 (Conv2d)
- weight形状: [out_channels, in_channels, kernel_height, kernel_width]
- 例如: \
Conv2d(3, 64, kernel_size=3)\
weight = torch.empty(64, 3, 3, 3)\
fan_in = 3 * 3 * 3 = 27   # in_channels * kernel_h * kernel_w\
fan_out = 64 * 3 * 3 = 576 # out_channels * kernel_h * kernel_w

- normal_() 方法\
用正态分布（高斯分布）的随机值填充张量。

- uniform_() 方法\
用均匀分布的随机值填充张量。

In [ ]:
import torch
import torch.nn as nn
from torch import Tensor
import math

def kaiming_init(weight: Tensor) -> Tensor:
    """
    使用Kaiming/He初始化方法初始化权重张量
    
    Args:
        weight (Tensor): 需要初始化的权重张量
        
    Returns:
        Tensor: 初始化后的权重张量（原地修改并返回）
    
    Note:
        对于ReLU激活函数，使用增益为 sqrt(2)
        对于LeakyReLU等变体，增益会有所不同
    """
    # 获取权重的维度信息
    if weight.dim() >= 2:
        # 对于卷积层或全连接层使用正态分布初始化
        fan_in, fan_out = nn.init._calculate_fan_in_and_fan_out(weight) # 根据不同的权重，例如 conv 或者 linear 找到输入和输出特征数
        
        # 使用ReLU的增益 sqrt(2)，**缩放因子补偿激活函数对信号方差的影响**
        gain = math.sqrt(2.0)
        
        # 计算标准差
        std = gain / math.sqrt(fan_in)
        
        with torch.no_grad():
            weight.normal_(0, std)
    else:
        # 对于一维权重（如偏置），使用简单的均匀分布
        with torch.no_grad():
            weight.uniform_(-1, 1)
    
    return weight


# 更简洁的实现版本（如果不需要复杂的维度检查）
def kaiming_init_simple(weight: Tensor) -> Tensor:
    """
    使用Kaiming初始化的简化版本
    直接使用 PyTorch 的 nn.init.kaiming_uniform_或kaiming_normal_
    """
    # 使用kaiming均匀分布初始化（默认使用ReLU）
    nn.init.kaiming_uniform_(weight, a=math.sqrt(5))
    return weight


# 带可选参数的完整版本
def kaiming_init_full(weight: Tensor, 
                      mode: str = 'fan_in', 
                      nonlinearity: str = 'relu',
                      distribution: str = 'normal') -> Tensor:
    """
    完整的Kaiming初始化函数
    
    Args:
        weight (Tensor): 需要初始化的权重张量
        mode (str): 'fan_in' 或 'fan_out'
        nonlinearity (str): 'relu' 或 'leaky_relu' 
        distribution (str): 'normal' 或 'uniform'
    
    Returns:
        Tensor: 初始化后的权重张量
    """
    if distribution == 'normal':
        nn.init.kaiming_normal_(weight, mode=mode, nonlinearity=nonlinearity)
    elif distribution == 'uniform':
        nn.init.kaiming_uniform_(weight, mode=mode, nonlinearity=nonlinearity)
    else:
        raise ValueError(f"不支持的分布类型: {distribution}")
    
    return weight


# 使用示例
if __name__ == "__main__":
    # 测试不同维度
    # 全连接层权重
    fc_weight = torch.empty(128, 256)
    kaiming_init(fc_weight)
    print(f"FC层权重均值: {fc_weight.mean():.4f}, 标准差: {fc_weight.std():.4f}")
    
    # 卷积层权重
    conv_weight = torch.empty(64, 3, 3, 3)  # [out_channels, in_channels, height, width]
    kaiming_init(conv_weight)
    print(f"卷积层权重均值: {conv_weight.mean():.4f}, 标准差: {conv_weight.std():.4f}")
    
    # 使用简化版本
    simple_weight = torch.empty(100, 50)
    kaiming_init_simple(simple_weight)
    print(f"简单版本权重均值: {simple_weight.mean():.4f}, 标准差: {simple_weight.std():.4f}")

In [ ]:
# 🧪 调试
import math
w = torch.empty(256, 512)
kaiming_init(w)
print(f'均值: {w.mean():.4f} (期望 ~0)')
print(f'标准差: {w.std():.4f} (期望 {math.sqrt(2/512):.4f})')

In [ ]:
# ✅ 提交
from torch_judge import check
check('weight_init')